# Session Management & Reactive State — Multigen SDK Notebook 26

**Multigen SDK — Notebook 26**

---

## What are sessions and reactive state?

In multi-agent systems, individual agents often need to maintain **identity**
across multiple calls — remembering which user they are serving, how long they
have been active, and when their context should expire.  The Multigen SDK provides
a full session management layer for exactly this purpose.

Alongside sessions, agents need **structured, validated, observable state**.  A bare
Python dict offers no safety guarantees.  Multigen's state subsystem adds:

- **Schema validation** — declare expected keys, types, and defaults up front.
- **Reactive watchers** — get notified the moment a specific key changes.
- **Computed (derived) properties** — define `total = price * qty` once;
  it re-evaluates automatically when either dependency changes.
- **State machine integration** — typed transitions with guards and side-effects.

## What this notebook covers

| Section | Topic |
|---------|-------|
| 1 | Setup and imports |
| 2 | Creating sessions — `SessionManager.create_session()` |
| 3 | Session lifecycle — touch, idle time, terminate, list_active |
| 4 | Session TTL and expiry |
| 5 | `with_session` async context manager |
| 6 | Session hooks — create and terminate callbacks |
| 7 | `SessionMiddleware` — auto-inject `_session` into agent context |
| 8 | `StateSchema` — required and optional fields |
| 9 | `StateInitializer` — create and merge with validator integration |
| 10 | `StateValidator` — require, type_check, range, custom rules |
| 11 | `ReactiveState` — watch multiple keys, update, snapshot |
| 12 | `ComputedState` — auto-recompute on dependency change |
| 13 | `StateTransition` + `StateMachineState` — guard, on_enter, on_exit, next_state |

All code is pure in-memory; no external services or databases required.


## Section 1 — Setup and Imports

We import from two SDK modules:

**`multigen.session`** — session lifecycle management:

| Class | Role |
|-------|------|
| `SessionState` | Enum: ACTIVE / IDLE / EXPIRED / TERMINATED |
| `SessionContext` | Dataclass holding session metadata and lifecycle state |
| `SessionManager` | High-level facade: create, touch, terminate, list, cleanup |
| `InMemorySessionStore` | Concrete store backed by a Python dict |
| `SessionMiddleware` | Wraps an agent function to auto-create/restore sessions |

**`multigen.state_init`** — state schema, validation, and reactivity:

| Class | Role |
|-------|------|
| `StateSchema` | Declare keys, types, and defaults for an agent state dict |
| `StateInitializer` | Factory: build a validated, fully-populated state dict |
| `ReactiveState` | Observable dict-like container; watchers fire on mutation |
| `ComputedState` | Derived properties that auto-recompute when dependencies change |
| `StateValidator` | Composable validation rules with error accumulation |
| `StateTransition` | Typed (from → to) transition with guard and side-effects |
| `StateMachineState` | Named state node with transitions for graph/state-machine use |


In [ ]:
import asyncio
import sys
import time

sys.path.insert(0, '../sdk')

from multigen.session import (
    SessionState,
    SessionContext,
    SessionManager,
    InMemorySessionStore,
    SessionMiddleware,
)

from multigen.state_init import (
    StateSchema,
    StateInitializer,
    ReactiveState,
    ComputedState,
    StateValidator,
    StateTransition,
    StateMachineState,
)

print('SessionState         :', SessionState)
print('SessionContext       :', SessionContext)
print('SessionManager       :', SessionManager)
print('InMemorySessionStore :', InMemorySessionStore)
print('SessionMiddleware    :', SessionMiddleware)
print()
print('StateSchema          :', StateSchema)
print('StateInitializer     :', StateInitializer)
print('ReactiveState        :', ReactiveState)
print('ComputedState        :', ComputedState)
print('StateValidator       :', StateValidator)
print('StateTransition      :', StateTransition)
print('StateMachineState    :', StateMachineState)
print()
print('All imports successful.')


## Section 2 — Creating Sessions

`SessionManager.create_session(agent_id)` creates a new `SessionContext` and
persists it in the underlying store.

A `SessionContext` carries:

| Field | Type | Description |
|-------|------|-------------|
| `session_id` | str | UUID auto-generated |
| `agent_id` | str | Which agent owns this session |
| `created_at` | float | Unix timestamp of creation |
| `last_active` | float | Unix timestamp of last touch |
| `state` | `SessionState` | ACTIVE / IDLE / EXPIRED / TERMINATED |
| `ttl` | float | Seconds before the session expires (from `last_active`) |
| `metadata` | dict | Arbitrary key-value data |
| `tags` | list[str] | Labels for querying |

Two computed properties:
- `is_active` — True if `state == ACTIVE` and not expired.
- `age` — seconds since `created_at`.


In [ ]:
store   = InMemorySessionStore()
manager = SessionManager(store=store, default_ttl=3600.0)

# Create a session for a research agent
session = await manager.create_session(
    "research_agent",
    metadata={"user": "alice", "task": "market analysis"},
    tags=["research", "premium"],
)

print('Session created:')
print(f'  session_id   : {session.session_id}')
print(f'  agent_id     : {session.agent_id}')
print(f'  state        : {session.state}')
print(f'  ttl          : {session.ttl} seconds')
print(f'  metadata     : {session.metadata}')
print(f'  tags         : {session.tags}')
print(f'  is_active    : {session.is_active}')
print(f'  age (sec)    : {session.age:.4f}')
print()

assert session.is_active, "Session should be active immediately after creation"
assert session.agent_id == "research_agent"
assert "user" in session.metadata

print('Session inspection verified.')


## Section 3 — Session Lifecycle

A session moves through these states:

```
ACTIVE  ──touch──►  ACTIVE  (last_active refreshed)
ACTIVE  ──idle──►   IDLE    (state transitions on inactivity)
IDLE    ──touch──►  ACTIVE  (re-activates)
ACTIVE  ──terminate──► TERMINATED
ACTIVE  ──expire──►  EXPIRED  (lazy, on next get())
```

Key operations:
- `touch_session(session_id)` — refresh `last_active`; promotes IDLE → ACTIVE.
- `terminate_session(session_id)` — mark as TERMINATED; fires terminate hooks.
- `list_active(agent_id=None)` — return all ACTIVE sessions.
- `cleanup()` — delete EXPIRED sessions from the store.


In [ ]:
lifecycle_manager = SessionManager(default_ttl=3600.0)

# Create two sessions for different agents
sess_a = await lifecycle_manager.create_session("agent_alpha", metadata={"role": "researcher"})
sess_b = await lifecycle_manager.create_session("agent_beta",  metadata={"role": "writer"})

print(f'Created session A: {sess_a.session_id[:8]}...  agent={sess_a.agent_id}')
print(f'Created session B: {sess_b.session_id[:8]}...  agent={sess_b.agent_id}')

# List active sessions
active_before = await lifecycle_manager.list_active()
print(f'\nActive sessions before terminate: {len(active_before)}  (expected: 2)')

# Touch session A (simulate activity)
touched = await lifecycle_manager.touch_session(sess_a.session_id)
refreshed_a = await lifecycle_manager.get_session(sess_a.session_id)
print(f'\ntouched session A: {touched}')
print(f'idle_time after touch: {refreshed_a.idle_time:.4f} sec  (should be near 0)')

# Terminate session B
terminated = await lifecycle_manager.terminate_session(sess_b.session_id)
refreshed_b = await lifecycle_manager.get_session(sess_b.session_id)
print(f'\nterminated session B: {terminated}')
print(f'session B state after terminate: {refreshed_b.state}  (expected: TERMINATED)')
print(f'session B is_active: {refreshed_b.is_active}  (expected: False)')

# List active sessions again — B should be gone
active_after = await lifecycle_manager.list_active()
print(f'\nActive sessions after terminate: {len(active_after)}  (expected: 1)')
print(f'Remaining active agent: {active_after[0].agent_id}  (expected: agent_alpha)')

assert len(active_after) == 1
assert active_after[0].agent_id == "agent_alpha"
print('\nSession lifecycle verified.')


## Section 4 — Session TTL and Expiry

Sessions created with a small `ttl` value (in seconds) automatically become
`EXPIRED` once `(now - last_active) > ttl`.

Expiry is **lazy**: the session is not deleted immediately.  Instead, when
`get_session()` is called on an expired session, the store marks its state as
`EXPIRED`.  Call `cleanup()` to physically remove expired sessions from the store.

This avoids background timers and keeps the implementation simple and testable.


In [ ]:
ttl_manager = SessionManager(default_ttl=3600.0)

# Create a session with a very short TTL (100 ms)
short_session = await ttl_manager.create_session("ttl_agent", ttl=0.1)
sid = short_session.session_id

print(f'Short-TTL session created: {sid[:8]}...')
print(f'  is_active immediately: {short_session.is_active}  (expected: True)')
print(f'  is_expired immediately: {short_session.is_expired}  (expected: False)')

# Wait for TTL to elapse
await asyncio.sleep(0.15)

# Fetch from store — lazy expiry marks the state as EXPIRED
fetched = await ttl_manager.get_session(sid)
print(f'\nAfter 150 ms sleep:')
print(f'  fetched.state     : {fetched.state}  (expected: EXPIRED)')
print(f'  fetched.is_expired: {fetched.is_expired}  (expected: True)')
print(f'  fetched.is_active : {fetched.is_active}   (expected: False)')

# cleanup() removes expired sessions
removed = await ttl_manager.cleanup()
print(f'\ncleanup() removed {removed} expired session(s)  (expected: 1)')

gone = await ttl_manager.get_session(sid)
print(f'get_session() after cleanup: {gone}  (expected: None)')

assert fetched.is_expired
assert removed == 1
assert gone is None
print('\nSession TTL and expiry verified.')


## Section 5 — `with_session` Async Context Manager

`SessionManager.with_session(agent_id)` is an async context manager that:

1. Creates a session on `__aenter__`.
2. Yields the `SessionContext` to the caller.
3. Terminates the session on `__aexit__` (even if an exception is raised).

This is the recommended pattern for short-lived agent invocations where you want
automatic cleanup and don't want to manage the session lifecycle manually.


In [ ]:
ctx_manager = SessionManager(default_ttl=60.0)

print('Entering with_session context...')

async with ctx_manager.with_session("context_agent", metadata={"task": "summarise"}) as ctx:
    print(f'  Inside context:')
    print(f'    session_id : {ctx.session_id[:8]}...')
    print(f'    agent_id   : {ctx.agent_id}')
    print(f'    is_active  : {ctx.is_active}')

    # Verify the session exists in the store while inside the context
    active_inside = await ctx_manager.list_active()
    print(f'    active sessions while inside context: {len(active_inside)}  (expected: 1)')
    assert len(active_inside) == 1
    inside_sid = ctx.session_id

print('\nExited with_session context.')

# After exiting, the session should be TERMINATED
session_after = await ctx_manager.get_session(inside_sid)
print(f'Session state after context exit: {session_after.state}  (expected: TERMINATED)')
print(f'is_active after context exit    : {session_after.is_active}  (expected: False)')

# No active sessions remain
active_after = await ctx_manager.list_active()
print(f'Active sessions after context exit: {len(active_after)}  (expected: 0)')

assert session_after.state.value == "terminated"
assert len(active_after) == 0
print('\nwith_session context manager verified.')


## Section 6 — Session Hooks

`SessionManager` supports async lifecycle hooks:

- `add_hook("create", async_fn)` — called after every `create_session()`.
- `add_hook("terminate", async_fn)` — called after every `terminate_session()`.

Each hook receives a `SessionEvent` dataclass with:
- `event_type` — `"create"` or `"terminate"`
- `session_id` — the affected session
- `timestamp` — float Unix timestamp
- `data` — dict with extra context (e.g. `{"agent_id": "..."}`)

Hooks are useful for audit logging, metrics emission, or cascading cleanup.


In [ ]:
hook_manager = SessionManager(default_ttl=3600.0)

create_events    = []
terminate_events = []

async def on_create(event):
    create_events.append(event)
    print(f'  [on_create hook]    session_id={event.session_id[:8]}...  data={event.data}')

async def on_terminate(event):
    terminate_events.append(event)
    print(f'  [on_terminate hook] session_id={event.session_id[:8]}...  data={event.data}')

hook_manager.add_hook("create",    on_create)
hook_manager.add_hook("terminate", on_terminate)

# Fire create hook
hook_sess = await hook_manager.create_session("hooked_agent")
print()

# Fire terminate hook
await hook_manager.terminate_session(hook_sess.session_id)
print()

print(f'create_events fired    : {len(create_events)}    (expected: 1)')
print(f'terminate_events fired : {len(terminate_events)} (expected: 1)')
print()
print(f'create event type      : {create_events[0].event_type!r}')
print(f'terminate event type   : {terminate_events[0].event_type!r}')
print(f'session_ids match      : {create_events[0].session_id == terminate_events[0].session_id}')

assert len(create_events) == 1
assert len(terminate_events) == 1
assert create_events[0].session_id == hook_sess.session_id
print('\nSession hooks verified.')


## Section 7 — SessionMiddleware

`SessionMiddleware` wraps any async agent function so that:

1. Before calling the agent, a session is looked up (by `ctx["session_id"]`) or
   a new one is created.
2. The `SessionContext` is injected into the call context as `ctx["_session"]`.
3. After the call, if a new session was created (i.e. no `session_id` was provided),
   it is automatically terminated.

This decouples session management from agent business logic entirely.


In [ ]:
mw_manager    = SessionManager(default_ttl=3600.0)
mw_middleware = SessionMiddleware(mw_manager)

# Simulated agent function — just inspects its context dict
async def my_agent_fn(ctx: dict) -> dict:
    session = ctx.get("_session")
    print(f'  [my_agent_fn] _session injected: {session is not None}')
    if session:
        print(f'  [my_agent_fn] session_id={session.session_id[:8]}...  '
              f'agent_id={session.agent_id!r}  is_active={session.is_active}')
    return {"processed": True, "session_id": session.session_id if session else None}

# --- Without pre-existing session_id (middleware creates one) ---
print('--- Calling agent without pre-existing session ---')
result = await mw_middleware(
    my_agent_fn,
    ctx={"agent_id": "middleware_agent", "input": "hello"},
)
print(f'  agent result: {result}')
print()

# Verify the session was terminated after the call
if result["session_id"]:
    post_session = await mw_manager.get_session(result["session_id"])
    if post_session:
        print(f'  session state after call: {post_session.state}  (expected: TERMINATED)')
        assert post_session.state.value == "terminated"

# --- With pre-existing session_id (middleware reuses it) ---
print('--- Calling agent with pre-existing session ---')
pre_session = await mw_manager.create_session("reuse_agent")
result2 = await mw_middleware(
    my_agent_fn,
    ctx={"agent_id": "reuse_agent",
         "session_id": pre_session.session_id,
         "input": "world"},
)
print(f'  agent result: {result2}')

# Pre-existing session should still be active (not auto-terminated)
pre_after = await mw_manager.get_session(pre_session.session_id)
print(f'  pre-existing session state after call: {pre_after.state}  (expected: ACTIVE or IDLE)')
assert pre_after.state in (SessionState.ACTIVE, SessionState.IDLE)

print('\nSessionMiddleware verified.')


## Section 8 — StateSchema

`StateSchema` declares the expected shape of an agent's state dict.

Each field is specified as:
- `key: type` — **required** field (no default).  `create()` raises if it is absent.
- `key: (type, default)` — **optional** field with a default value.

The schema exposes:
- `required_keys()` — list of keys with no default.
- `optional_keys()` — list of keys that have a default.
- `default_for(key)` — returns a fresh copy of the default value (safe for
  mutable defaults like `list` and `dict`).
- `type_for(key)` — the declared Python type.


In [ ]:
schema = StateSchema({
    # Required fields (no default)
    "query":    str,
    "agent_id": str,
    # Optional fields (type, default)
    "results":  (list,  None),    # None → [] via _coerce_default
    "retries":  (int,   0),
    "done":     (bool,  False),
    "metadata": (dict,  None),    # None → {}
    "tags":     (list,  []),
})

print('Schema overview:')
print(f'  All keys      : {schema.keys()}')
print(f'  Required keys : {schema.required_keys()}')
print(f'  Optional keys : {schema.optional_keys()}')
print()

print('Defaults for optional keys:')
for key in schema.optional_keys():
    print(f'  {key:<10}  type={schema.type_for(key).__name__:<8}  '
          f'default={schema.default_for(key)!r}')

print()
# Verify mutable defaults are fresh copies (not shared references)
list_default_1 = schema.default_for("results")
list_default_2 = schema.default_for("results")
list_default_1.append("mutated")
print(f'Mutable default independence test:')
print(f'  list_default_1 (after mutation) : {list_default_1}')
print(f'  list_default_2 (fresh copy)     : {list_default_2}')
assert list_default_2 == [], "Mutable defaults should be independent copies"

print('\nStateSchema verified.')


## Section 9 — StateInitializer

`StateInitializer` is the factory that turns a `StateSchema` into a fully-populated,
validated state dict.

Key methods:
- `create(overrides=None)` — start from schema defaults, apply overrides, then
  run the attached validator (if any).  Raises `ValueError` on validation failure.
- `merge(base, patch)` — deep-merge `patch` into `base`; nested dicts are merged
  recursively rather than replaced wholesale.


In [ ]:
# Build a schema with a validator that requires "query"
v = StateValidator()
v.require("query")
v.type_check("retries", int)

agent_schema = StateSchema({
    "query":   str,
    "results": (list, None),
    "retries": (int,  0),
    "done":    (bool, False),
})

initializer = StateInitializer(agent_schema, validator=v)

# --- create() with overrides ---
state = initializer.create({"query": "AI safety research"})
print('State created with overrides:')
for k, val in state.items():
    print(f'  {k:<10}: {val!r}')

assert state["query"]   == "AI safety research"
assert state["retries"] == 0
assert state["done"]    is False
assert state["results"] == []

print()

# --- create() without required field raises ---
try:
    initializer.create({})   # "query" is required
    print("ERROR: should have raised ValueError")
except ValueError as exc:
    print(f'create() without required field raised ValueError as expected:')
    print(f'  {exc}')

print()

# --- merge() deep-merges nested dicts ---
base  = {"query": "hello", "metadata": {"env": "prod", "retries": 0}}
patch = {"metadata": {"retries": 3, "debug": True}}
merged = initializer.merge(base, patch)

print('merge() result:')
print(f'  {merged}')

assert merged["metadata"]["env"]     == "prod"      # preserved from base
assert merged["metadata"]["retries"] == 3            # patched
assert merged["metadata"]["debug"]   is True         # added by patch

print('\nStateInitializer.create() and merge() verified.')


## Section 10 — StateValidator

`StateValidator` accumulates validation rules and checks a state dict against all of
them in one pass.  Rules:

| Method | Behaviour |
|--------|-----------|
| `require(key)` | Fail if key is absent or `None` |
| `type_check(key, type)` | Fail if value is present but wrong type |
| `range(key, min_val, max_val)` | Fail if numeric value is out of range |
| `custom(key, predicate, message)` | Fail if `predicate(value)` returns False |

`validate(state)` returns a list of `ValidationError` objects — one per failed rule.
`validate_raise(state)` raises `ValueError` with all error messages concatenated if
any rule fails.


In [ ]:
v2 = StateValidator()
v2.require("query")
v2.require("agent_id")
v2.type_check("retries", int)
v2.range("retries", min_val=0, max_val=10)
v2.custom("query", lambda q: len(q) >= 3, "query must be at least 3 characters")

# --- Valid state ---
valid_state = {"query": "market analysis", "agent_id": "agent_01", "retries": 2}
errors_valid = v2.validate(valid_state)
print(f'Valid state — errors: {errors_valid}  (expected: [])')
assert errors_valid == []

# --- State with multiple violations ---
bad_state = {
    "query":    "ab",       # too short (< 3 chars)
    # agent_id: missing     — violates require
    "retries":  "five",     # wrong type
}
errors_bad = v2.validate(bad_state)
print(f'\nInvalid state — errors ({len(errors_bad)}):')
for err in errors_bad:
    print(f'  key={err.key!r:<12}  message={err.message!r}')

# validate_raise accumulates all errors into one ValueError
try:
    v2.validate_raise(bad_state)
    print("ERROR: should have raised ValueError")
except ValueError as exc:
    print(f'\nvalidate_raise() raised ValueError:')
    print(f'  {exc}')

# --- Range violation ---
range_state = {"query": "hello", "agent_id": "a", "retries": 15}
range_errors = v2.validate(range_state)
print(f'\nRange violation state — errors: {len(range_errors)}')
for err in range_errors:
    print(f'  key={err.key!r:<12}  message={err.message!r}')

assert len(errors_bad) >= 3, f"Expected >= 3 errors, got {len(errors_bad)}"
assert len(range_errors) == 1
print('\nStateValidator verified.')


## Section 11 — ReactiveState

`ReactiveState` is an observable dict-like container.  Register watchers on specific
keys; they fire whenever the value changes.

```python
rs = ReactiveState({"count": 0})
rs.watch("count", lambda old, new: print(f"count: {old} → {new}"))
rs["count"] = 5   # fires watcher: "count: 0 → 5"
```

Both sync and async watchers are supported.  Async watchers are scheduled as
`asyncio.create_task` so they don't block the setter.

`rs.snapshot()` returns a deep copy of the current state — safe to persist or
diff without worrying about concurrent mutations.


In [ ]:
rs = ReactiveState({"status": "idle", "count": 0, "items": []})

watcher_log = []

# Sync watcher on "status"
def on_status_change(old, new):
    watcher_log.append(("status", old, new))
    print(f'  [watcher:status] {old!r} → {new!r}')

# Async watcher on "count"
async def on_count_change(old, new):
    watcher_log.append(("count", old, new))
    print(f'  [watcher:count]  {old!r} → {new!r}')

rs.watch("status", on_status_change)
rs.watch("count",  on_count_change)

print('--- Triggering state changes ---')
rs["status"] = "running"    # fires on_status_change
rs["count"]  = 1             # fires on_count_change (as task)
rs["count"]  = 2             # fires on_count_change (as task)
rs["items"]  = ["a", "b"]   # no watcher — silent

# Yield control so async watchers can complete
await asyncio.sleep(0)

print()
print(f'Watcher log: {watcher_log}')

# Snapshot captures current state
snap = rs.snapshot()
print(f'\nSnapshot: {snap}')

# Mutate rs AFTER snapshot — snapshot should not change
rs["count"] = 99
print(f'rs["count"] after mutation : {rs["count"]}')
print(f'snapshot["count"] untouched: {snap["count"]}  (expected: 2)')

assert snap["count"] == 2, "Snapshot should be independent of subsequent mutations"
assert rs["status"] == "running"

# update() sets multiple keys at once
rs.update({"status": "done", "count": 0})

await asyncio.sleep(0)   # let async watchers run

print(f'\nFinal ReactiveState: {rs}')
print('\nReactiveState watchers and snapshot verified.')


## Section 12 — ComputedState

`ComputedState` wraps a `ReactiveState` and adds **derived properties** that
automatically re-compute when any of their declared dependencies change.

Computation is **lazy**: the function is not called until you read the computed
property.  Once computed, the result is cached until a dependency changes (marked
`dirty`).

```python
cs.define("total", ["price", "qty"], lambda s: s["price"] * s["qty"])
print(cs["total"])   # computes: 10.0 * 3 = 30.0
rs["price"] = 20.0  # marks "total" as dirty
print(cs["total"])   # recomputes: 20.0 * 3 = 60.0
```


In [ ]:
rs_shop = ReactiveState({"price": 10.0, "qty": 3, "discount": 0.0})
cs = ComputedState(rs_shop)

# Define "total" — depends on price and qty
cs.define(
    "total",
    depends_on=["price", "qty"],
    fn=lambda s: round(s["price"] * s["qty"], 2),
)

# Define "discounted_total" — depends on price, qty, and discount
cs.define(
    "discounted_total",
    depends_on=["price", "qty", "discount"],
    fn=lambda s: round(s["price"] * s["qty"] * (1.0 - s["discount"]), 2),
)

print(f'Initial price={rs_shop["price"]}  qty={rs_shop["qty"]}  discount={rs_shop["discount"]}')
print(f'  cs["total"]             = {cs["total"]}             (expected: 30.0)')
print(f'  cs["discounted_total"]  = {cs["discounted_total"]}  (expected: 30.0)')

# Update price — total should recompute
rs_shop["price"] = 20.0
print(f'\nAfter price → 20.0:')
print(f'  cs["total"]             = {cs["total"]}             (expected: 60.0)')
print(f'  cs["discounted_total"]  = {cs["discounted_total"]}  (expected: 60.0)')

assert cs["total"] == 60.0
assert cs["discounted_total"] == 60.0

# Apply a 10% discount
rs_shop["discount"] = 0.10
print(f'\nAfter discount → 0.10:')
print(f'  cs["discounted_total"]  = {cs["discounted_total"]}  (expected: 54.0)')
print(f'  cs["total"] unchanged   = {cs["total"]}             (expected: 60.0, discount not in deps)')

assert cs["discounted_total"] == 54.0
assert cs["total"] == 60.0   # "total" does not depend on "discount"

print(f'\nDefined computed keys: {cs.keys()}')
print('\nComputedState auto-recompute verified.')


## Section 13 — StateTransition and StateMachineState

The `state_init` module provides building blocks for typed state machines
independent of the probabilistic `StateMachine` class:

**`StateTransition`** — a typed `(from_state → to_state)` edge with:
- `guard` — optional `Callable[[ctx], bool]`; transition is blocked if False.
- `on_exit` — called on the departing state before the transition.
- `on_enter` — called on the arriving state after the transition.
- `can_apply(ctx)` — returns True if the guard passes (or no guard is set).
- `apply(ctx)` — fires on_exit, sets `ctx["_state"] = to_state`, fires on_enter.

**`StateMachineState`** — a named node that holds a list of outgoing
`StateTransition` objects:
- `add_transition(t)` — register an outgoing transition.
- `next_state(ctx)` — return the `to_state` of the first applicable transition,
  or `None` if no transition passes its guard.


In [ ]:
side_effect_log = []

# Define transitions
t_idle_to_running = StateTransition(
    from_state="idle",
    to_state="running",
    guard=lambda ctx: ctx.get("ready", False),
    on_exit=lambda ctx: side_effect_log.append(("exit", "idle")),
    on_enter=lambda ctx: side_effect_log.append(("enter", "running")),
    label="start",
)

t_running_to_done = StateTransition(
    from_state="running",
    to_state="done",
    guard=lambda ctx: ctx.get("result") is not None,
    on_exit=lambda ctx: side_effect_log.append(("exit", "running")),
    on_enter=lambda ctx: side_effect_log.append(("enter", "done")),
    label="finish",
)

t_running_to_error = StateTransition(
    from_state="running",
    to_state="error",
    guard=lambda ctx: ctx.get("error") is not None,
    label="fail",
)

# Build StateMachineState nodes
idle_node    = StateMachineState(name="idle",    is_terminal=False)
running_node = StateMachineState(name="running", is_terminal=False)
done_node    = StateMachineState(name="done",    is_terminal=True)

idle_node.add_transition(t_idle_to_running)
running_node.add_transition(t_running_to_done)
running_node.add_transition(t_running_to_error)

# ── Test can_apply / guard behaviour ─────────────────────────────────────────
ctx_not_ready = {"ready": False}
ctx_ready     = {"ready": True}

print('Guard tests:')
print(f'  t_idle_to_running.can_apply(ready=False): {t_idle_to_running.can_apply(ctx_not_ready)}  (expected: False)')
print(f'  t_idle_to_running.can_apply(ready=True) : {t_idle_to_running.can_apply(ctx_ready)}  (expected: True)')

assert not t_idle_to_running.can_apply(ctx_not_ready)
assert     t_idle_to_running.can_apply(ctx_ready)

# ── Test apply() fires on_exit / on_enter ─────────────────────────────────────
print()
ctx_running = await t_idle_to_running.apply({"ready": True})
print(f'Context after idle→running transition:')
print(f'  _state        : {ctx_running["_state"]}  (expected: "running")')
print(f'  side_effects  : {side_effect_log}')

assert ctx_running["_state"] == "running"
assert ("exit",  "idle")    in side_effect_log
assert ("enter", "running") in side_effect_log

# ── Test next_state() dispatch ────────────────────────────────────────────────
print()
ctx_with_result = {"result": "summary complete", "error": None}
ctx_with_error  = {"result": None, "error": "timeout"}
ctx_neither     = {"result": None, "error": None}

ns_done  = running_node.next_state(ctx_with_result)
ns_error = running_node.next_state(ctx_with_error)
ns_none  = running_node.next_state(ctx_neither)

print('next_state() dispatch from "running":')
print(f'  ctx has result → next_state = {ns_done!r}   (expected: "done")')
print(f'  ctx has error  → next_state = {ns_error!r}  (expected: "error")')
print(f'  ctx has neither→ next_state = {ns_none!r}   (expected: None)')

assert ns_done  == "done"
assert ns_error == "error"
assert ns_none  is None

print()
print(f'done_node.is_terminal: {done_node.is_terminal}  (expected: True)')
assert done_node.is_terminal

print('\nStateTransition and StateMachineState verified.')


## Summary

### Session Management API at a glance

| Method | Signature | What it does |
|--------|-----------|-------------|
| `create_session` | `(agent_id, ttl=None, metadata={}, tags=[])` | Create and persist a new session |
| `get_session` | `(session_id)` | Retrieve session; lazily marks as EXPIRED if TTL elapsed |
| `touch_session` | `(session_id)` | Refresh `last_active`; promotes IDLE → ACTIVE |
| `terminate_session` | `(session_id)` | Mark as TERMINATED; fires terminate hooks |
| `list_active` | `(agent_id=None)` | Return all ACTIVE sessions |
| `cleanup` | `()` | Delete EXPIRED sessions from the store |
| `with_session` | `(agent_id)` | Async context manager: create on enter, terminate on exit |
| `add_hook` | `("create"\|"terminate", async_fn)` | Register lifecycle callback |

### Session state machine

```
create_session()   →  ACTIVE
touch_session()    →  ACTIVE  (from IDLE)
TTL elapsed        →  EXPIRED  (lazy, on next get())
terminate_session()→  TERMINATED
cleanup()          →  removes EXPIRED from store
```

### Reactive State API at a glance

| Class | Key API |
|-------|---------|
| `StateSchema` | `required_keys()`, `optional_keys()`, `default_for(key)` |
| `StateInitializer` | `create(overrides)`, `merge(base, patch)` |
| `StateValidator` | `require()`, `type_check()`, `range()`, `custom()`, `validate()`, `validate_raise()` |
| `ReactiveState` | `watch(key, fn)`, `__setitem__`, `snapshot()`, `update()` |
| `ComputedState` | `define(name, depends_on, fn)`, `__getitem__` |
| `StateTransition` | `can_apply(ctx)`, `apply(ctx)` — fires guards and side-effects |
| `StateMachineState` | `add_transition(t)`, `next_state(ctx)` |

### Next steps

- **Notebook 25**: Advanced Agent Messaging (`AdvancedMessageBus`, priority, DLQ)
- **Notebook 16**: Probabilistic state machines with MCMC sampling
- Combine `ReactiveState` with `AdvancedMessageBus` to broadcast state changes
  as messages to interested agents
- Use `SessionMiddleware` in a `Chain` pipeline to automatically scope every
  step to the correct session context
